# 🎥 06. Kaggle Notebook: X3D-S Video Classifier Fine-tuning
Fine-tune mô hình X3D-S (Pretrained Kinetics-400) trên các đoạn clip 16-frames để phân biệt vụ va chạm thực tế và phanh gấp thông thường.

In [ ]:
import torch
import torch.nn as nn
from src.classifiers.video_classifier import VideoAccidentClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
classifier = VideoAccidentClassifier(device=device, num_frames=16, crop_size=160)
print("Visual Classifier Architecture Loaded:", type(classifier.model))

## 1. Cấu hình Freeze Backbone & Train Head
Chỉ huấn luyện các block cuối và classification projection để tiết kiệm VRAM và tăng tốc hội tụ.

In [ ]:
# Đóng băng các lớp đầu
trainable_params = []
for name, param in classifier.model.named_parameters():
    if "classifier" in name or "proj" in name or "blocks.5" in name:
        param.requires_grad = True
        trainable_params.append(param)
    else:
        param.requires_grad = False

print(f"Trainable parameter tensors: {len(trainable_params)}")
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-3)

## 2. Thử nghiệm Forward Pass trên Dummy Video Clip

In [ ]:
import numpy as np
# 16 frames ảnh kích thước 160x160
dummy_clip = [np.random.randint(0, 255, (160, 160, 3), dtype=np.uint8) for _ in range(16)]
prob = classifier.predict(dummy_clip)
print(f"Inference output accident probability: {prob:.4f}")